# HCP Diffusion MRI Preprocessing Pipeline (Kaggle version)

Adapted from the Colab notebook. Before running:
- Settings panel (right sidebar) -> **Internet: On**
- Settings panel -> **Accelerator: GPU T4 x2** or **GPU P100**
- Add-ons -> Secrets -> add `HCP_AWS_ACCESS_KEY` and `HCP_AWS_SECRET_KEY`

## Step 1 - GPU check

In [ ]:
!nvidia-smi

## Step 2 - Install MRtrix3

In [ ]:
!apt-get -qq update
!apt-get -qq install -y mrtrix3
!mrconvert -version


## Step 3 - Install HD-BET

In [ ]:
!pip install -q git+https://github.com/MIC-DKFZ/HD-BET.git


## Step 4 - Install pinned scientific stack

Same exact pins as the HCP and ADNI Colab notebooks, so both datasets are processed by identical tensor-fitting logic.

In [ ]:
!pip install -q --no-cache-dir --only-binary=:all: "numpy==1.26.4" "scipy==1.13.1" "dipy==1.9.0" "nibabel==5.2.1"
!pip check


## → Restart now

Run menu -> **Restart & Clear Cell Output**. Then continue below in a fresh run — do NOT re-run Steps 1-4.

## Step 5 - Verify environment

In [ ]:
import numpy as np
import scipy
import dipy
import nibabel as nib

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("dipy:", dipy.__version__)
print("nibabel:", nib.__version__)

print("numpy compute check:", np.zeros(5).sum())

from dipy.segment.mask import median_otsu
test = np.random.rand(10, 10, 10)
_ = median_otsu(test, median_radius=1, numpass=1)
print("\ndipy/numba sanity check passed - environment is ready.")


## Step 6 - Subject list and paths

In [ ]:
import os, time, shutil, subprocess

SUBJECTS = [
    "100307", "100408", "101107", "101309", "101915",
    "102008", "102311", "102816", "103111", "103414",
    "103818", "104012", "104416", "105014", "105115",
    "105216", "105923", "106016", "106319", "106521",
    '107321', '107422', '108121', '108323', '108525', '108828', '109123', '109325', '110411', '111312', '111413', '111514', '111716', '112819', '113215', '113619', '113821', '113922', '114419', '114924', '115320', '116120', '116524', '117122', '117324', '118528', '118730'
]

RAW_DIR = "/kaggle/tmp/hcp_raw"  # scratch space, not counted against the 20GB /kaggle/working cap
OUT_DIR = "/kaggle/working/processed_hcp"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


## Step 6b - AWS credentials (via Kaggle Secrets)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    aws_access_key = secrets.get_secret("HCP_AWS_ACCESS_KEY").strip()
    aws_secret_key = secrets.get_secret("HCP_AWS_SECRET_KEY").strip()
    print("Loaded AWS credentials from Kaggle Secrets.")
except Exception as e:
    print(f"Could not load from Kaggle Secrets ({e}). Falling back to manual entry.")
    import getpass
    aws_access_key = getpass.getpass("AWS Access Key ID: ").strip()
    aws_secret_key = getpass.getpass("AWS Secret Access Key: ").strip()

os.makedirs(os.path.expanduser("~/.aws"), exist_ok=True)
with open(os.path.expanduser("~/.aws/credentials"), "w") as f:
    f.write(f"[default]\naws_access_key_id = {aws_access_key}\naws_secret_access_key = {aws_secret_key}\n")
with open(os.path.expanduser("~/.aws/config"), "w") as f:
    f.write("[default]\nregion = us-east-1\noutput = json\n")

print("AWS credentials configured.")

# Sanity check the file is clean (no embedded newlines / stray characters)
with open(os.path.expanduser("~/.aws/credentials")) as f:
    print(repr(f.read()))


## Step 7 - Install awscli and confirm S3 access

In [ ]:
!pip install -q awscli

result = subprocess.run(
    "aws s3 ls s3://hcp-openaccess/HCP/ --request-payer requester",
    shell=True, capture_output=True, text=True
)
lines = result.stdout.strip().split("\n")
print("First 3 subject folders found:")
print("\n".join(lines[:3]))

if result.returncode != 0:
    print("\nSTDERR (credential/access issue if this is non-empty):")
    print(result.stderr[:1000])
else:
    print(f"\nAccess confirmed - {len(lines)} total entries visible.")


In [ ]:
result = subprocess.run(
    "aws s3 ls s3://hcp-openaccess/HCP/ --request-payer requester",
    shell=True, capture_output=True, text=True
)
lines = result.stdout.strip().split("\n")

# Each line looks like: "                           PRE 100307/"
# Extract just the numeric subject ID
subject_ids = [line.split("PRE ")[-1].strip("/").strip() for line in lines if "PRE" in line]

print(f"Total subjects found: {len(subject_ids)}")
print("First 10:", subject_ids[:10])

SUBJECTS = subject_ids[:50]
print(f"\nUsing first 50 subjects:")
print(SUBJECTS)

## Step 7b - Pipeline functions

`download_subject` pulls the T1 and diffusion series for one subject from S3.
`preprocess_subject` subsets to a single shell, denoises, skull-strips, and fits the tensor to produce FA/MD.

In [ ]:
import os
import time
import subprocess
import numpy as np
import nibabel as nib
from dipy.io import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.dti import TensorModel


def is_valid_nifti(path):
    """Returns True only if the file exists AND can actually be read in full.
    Catches truncated/corrupt .nii.gz files left behind by interrupted runs,
    which a plain os.path.exists() check would incorrectly treat as 'done'."""
    if not os.path.exists(path):
        return False
    try:
        img = nib.load(path)
        img.get_fdata()  # forces a full read through the compressed data
        return True
    except Exception:
        return False


def download_subject(subj, dest_dir):
    t1_dir = os.path.join(dest_dir, "T1w")
    dwi_dir = os.path.join(t1_dir, "Diffusion")
    os.makedirs(dwi_dir, exist_ok=True)

    files_to_download = [
        (f"s3://hcp-openaccess/HCP/{subj}/T1w/T1w_acpc_dc_restore.nii.gz",
         os.path.join(t1_dir, "T1w_acpc_dc_restore.nii.gz")),
        (f"s3://hcp-openaccess/HCP/{subj}/T1w/Diffusion/data.nii.gz",
         os.path.join(dwi_dir, "data.nii.gz")),
        (f"s3://hcp-openaccess/HCP/{subj}/T1w/Diffusion/bvals",
         os.path.join(dwi_dir, "bvals")),
        (f"s3://hcp-openaccess/HCP/{subj}/T1w/Diffusion/bvecs",
         os.path.join(dwi_dir, "bvecs")),
        (f"s3://hcp-openaccess/HCP/{subj}/T1w/Diffusion/nodif_brain_mask.nii.gz",
         os.path.join(dwi_dir, "nodif_brain_mask.nii.gz")),
    ]

    for s3_path, local_path in files_to_download:
        # bvals/bvecs are plain text, not nifti - keep a simple existence check for those
        if local_path.endswith((".bval", "bvals", ".bvec", "bvecs")):
            if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
                continue
        else:
            if is_valid_nifti(local_path):
                continue
        cmd = f'aws s3 cp "{s3_path}" "{local_path}" --request-payer requester'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Download failed for {s3_path}: {result.stderr[:300]}")


def preprocess_subject(subj, raw_dir, out_dir, use_gpu=True):
    t_start = time.time()
    subj_raw = os.path.join(raw_dir, subj, "T1w")
    subj_out = os.path.join(out_dir, subj)
    os.makedirs(subj_out, exist_ok=True)

    t1_path    = os.path.join(subj_raw, "T1w_acpc_dc_restore.nii.gz")
    dwi_path   = os.path.join(subj_raw, "Diffusion", "data.nii.gz")
    bvals_path = os.path.join(subj_raw, "Diffusion", "bvals")
    bvecs_path = os.path.join(subj_raw, "Diffusion", "bvecs")
    mask_path  = os.path.join(subj_raw, "Diffusion", "nodif_brain_mask.nii.gz")

    subset_path   = os.path.join(subj_out, "data_subset.nii.gz")
    denoised_path = os.path.join(subj_out, "data_denoised.nii.gz")
    t1_brain_path = os.path.join(subj_out, "T1_brain.nii.gz")

    # 1/3: Subset to a single shell (b=0,1000) to keep runtime manageable
    if is_valid_nifti(subset_path):
        print(f"  [{subj}] 1/3: subset already exists and is valid - skipping.")
    else:
        print(f"  [{subj}] 1/3: Subsetting to single shell (dwiextract)...")
        t0 = time.time()
        cmd = (
            f'dwiextract "{dwi_path}" "{subset_path}" '
            f'-fslgrad "{bvecs_path}" "{bvals_path}" -shells 0,1000 -force'
        )
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0 or not is_valid_nifti(subset_path):
            if os.path.exists(subset_path):
                os.remove(subset_path)
            raise RuntimeError(f"dwiextract failed: {r.stderr[-500:]}")
        print(f"    done in {time.time()-t0:.1f}s")

    # 2/3: Denoise
    if is_valid_nifti(denoised_path):
        print(f"  [{subj}] 2/3: denoised already exists and is valid - skipping.")
    else:
        print(f"  [{subj}] 2/3: dwidenoise...")
        t0 = time.time()
        cmd = f'dwidenoise "{subset_path}" "{denoised_path}" -force'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0 or not is_valid_nifti(denoised_path):
            if os.path.exists(denoised_path):
                os.remove(denoised_path)
            raise RuntimeError(f"dwidenoise failed: {r.stderr[-500:]}")
        print(f"    done in {time.time()-t0:.1f}s")

    # 3/3: Skull-strip T1 - HD-BET primary, HCP's own precomputed brain as fallback
    if is_valid_nifti(t1_brain_path):
        print(f"  [{subj}] 3/3: T1_brain already exists and is valid - skipping.")
    else:
        print(f"  [{subj}] 3/3: HD-BET skull strip...")
        t0 = time.time()
        device = "cuda" if use_gpu else "cpu"
        cmd = (
            f'hd-bet -i "{t1_path}" -o "{t1_brain_path}" -device {device} '
            f'--disable_tta'
        )
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0 or not is_valid_nifti(t1_brain_path):
            print(f"    HD-BET failed ({r.stderr[-300:].strip()}), falling back to HCP's precomputed skull-stripped T1...")
            if os.path.exists(t1_brain_path):
                os.remove(t1_brain_path)
            fallback_cmd = (
                f'aws s3 cp "s3://hcp-openaccess/HCP/{subj}/T1w/T1w_acpc_dc_restore_brain.nii.gz" '
                f'"{t1_brain_path}" --request-payer requester'
            )
            rf = subprocess.run(fallback_cmd, shell=True, capture_output=True, text=True)
            if rf.returncode != 0 or not is_valid_nifti(t1_brain_path):
                raise RuntimeError(f"Both HD-BET and HCP fallback brain download failed: {rf.stderr[-500:]}")
            print(f"    fallback succeeded (used HCP's precomputed brain-extracted T1).")
        print(f"    done in {time.time()-t0:.1f}s")

    # Tensor fit -> FA, MD, tensor
    fa_path     = os.path.join(subj_out, "FA.nii.gz")
    md_path     = os.path.join(subj_out, "MD.nii.gz")
    tensor_path = os.path.join(subj_out, "tensor.nii.gz")
    mask_out    = os.path.join(subj_out, "brain_mask.nii.gz")

    if all(is_valid_nifti(p) for p in (fa_path, md_path, tensor_path, mask_out)):
        print(f"  [{subj}] Tensor outputs already exist and are valid - skipping.")
    else:
        print(f"  [{subj}] Fitting diffusion tensor...")
        t0 = time.time()

        dwi_img = nib.load(denoised_path)
        dwi_data = dwi_img.get_fdata()
        mask_data = nib.load(mask_path).get_fdata().astype(bool)

        full_bvals, full_bvecs = read_bvals_bvecs(bvals_path, bvecs_path)
        shell_mask = (np.abs(full_bvals - 0) < 100) | (np.abs(full_bvals - 1000) < 100)
        bvals = full_bvals[shell_mask]
        bvecs = full_bvecs[shell_mask]

        if bvals.shape[0] != dwi_data.shape[-1]:
            raise RuntimeError(
                f"Gradient/volume count mismatch after shell subsetting: "
                f"{bvals.shape[0]} gradients vs {dwi_data.shape[-1]} volumes"
            )

        gtab = gradient_table(bvals, bvecs)

        tenmodel = TensorModel(gtab)
        tenfit = tenmodel.fit(dwi_data, mask=mask_data)

        nib.save(nib.Nifti1Image(tenfit.fa.astype(np.float32), dwi_img.affine), fa_path)
        nib.save(nib.Nifti1Image(tenfit.md.astype(np.float32), dwi_img.affine), md_path)
        nib.save(nib.Nifti1Image(tenfit.lower_triangular().astype(np.float32), dwi_img.affine), tensor_path)
        nib.save(nib.Nifti1Image(mask_data.astype(np.uint8), dwi_img.affine), mask_out)
        print(f"    done in {time.time()-t0:.1f}s")

    for f in (subset_path, denoised_path):
        if os.path.exists(f):
            os.remove(f)

    print(f"  [{subj}] TOTAL TIME: {time.time()-t_start:.1f}s. Saved to {subj_out}")

In [ ]:
# Standalone HD-BET diagnostic test — run this before re-attempting Step 8

test_subj = "100307"
test_raw_dir = os.path.join(RAW_DIR, test_subj)

# Make sure raw files exist (redownload if the previous failed run deleted them)
t1_test_path = os.path.join(test_raw_dir, "T1w", "T1w_acpc_dc_restore.nii.gz")
if not os.path.exists(t1_test_path):
    print(f"Raw files for {test_subj} not found, redownloading...")
    download_subject(test_subj, test_raw_dir)
    print("Download complete.")
else:
    print(f"Raw files for {test_subj} already present.")

out_path = "/kaggle/working/test_brain.nii.gz"
cmd = f'hd-bet -i "{t1_test_path}" -o "{out_path}" -device cuda --disable_tta'

print(f"\nRunning: {cmd}\n")
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT (last 2000 chars) ---")
print(result.stdout[-2000:])
print("\n--- STDERR (last 2000 chars) ---")
print(result.stderr[-2000:])

if result.returncode == 0 and os.path.exists(out_path):
    print("\n✅ HD-BET succeeded — output file created.")
else:
    print("\n❌ HD-BET failed — see STDERR above for the real error.")

In [ ]:
# === New cell: select next batch of new HCP subjects, no overlap ===
import subprocess

# 1. Pull the CURRENT full subject listing directly from S3 (same call as Cell 17/18)
result = subprocess.run(
    "aws s3 ls s3://hcp-openaccess/HCP/ --request-payer requester",
    shell=True, capture_output=True, text=True
)
lines = result.stdout.strip().split("\n")
all_subject_ids = [line.split("PRE ")[-1].strip("/").strip() for line in lines if "PRE" in line]
print(f"Total subjects currently visible in bucket: {len(all_subject_ids)}")

# 2. Subjects already processed (confirm this list against your paper's real 40/41 before running)
ALREADY_USED = [
    '100307', '100408', '101006', '101107', '101309', '101410', '101915', '102008',
    '102311', '102816', '103111', '103414', '103515', '103818', '104820', '105014',
    '105115', '105216', '106016', '106319', '106521', '107321', '107422', '108121',
    '108323', '108525', '108828', '109123', '110411', '111312', '111413', '111716',
    '112819', '113215', '113619', '113821', '113922', '114419', '115320', '116524',
    '117122',
]
# Known-bad IDs from your last run (S3 404s) — exclude so you don't re-hit the same failures
KNOWN_FAILED = ['104012', '105923', '109325', '111514', '114924', '116120']

exclude = set(ALREADY_USED) | set(KNOWN_FAILED)
candidates = [s for s in all_subject_ids if s not in exclude]

TARGET_TOTAL = 253
n_needed = TARGET_TOTAL - len(ALREADY_USED)
print(f"Already used: {len(ALREADY_USED)}  |  Need {n_needed} new subjects to reach {TARGET_TOTAL}")

NEW_SUBJECTS = candidates[:n_needed]
print(f"Selected {len(NEW_SUBJECTS)} new candidate IDs (first 10 shown):")
print(NEW_SUBJECTS[:10])

# Use NEW_SUBJECTS as your SUBJECTS list in Step 8 — same resume/succeeded/failed
# tracking your notebook already has will handle any of these that also fail to download.

## Step 8 - Run the full pipeline

Includes a resume check: subjects with an existing `FA.nii.gz` are skipped, so restarting the session after a disconnect (e.g. GPU limit) does not redo finished work.

Suggestion: before looping over all 50, test on just the first 1-2 subjects to confirm timing and that nothing crashes. To do that, temporarily change `SUBJECTS` in Step 6 to `SUBJECTS[:2]`, run this cell, confirm it works, then change it back to the full list and re-run.

In [ ]:
import os
print("OUT_DIR:", OUT_DIR)
print("Exists:", os.path.exists(OUT_DIR))
print("Contents:", os.listdir(OUT_DIR) if os.path.exists(OUT_DIR) else "N/A")

# Check specifically for one of your 11 finished subjects
test_subj = NEW_SUBJECTS[0]  # or type one of your known finished subject IDs directly
test_path = os.path.join(OUT_DIR, test_subj, "FA.nii.gz")
print(f"\nChecking: {test_path}")
print("Exists:", os.path.exists(test_path))

In [ ]:
import os

for subj_folder in sorted(os.listdir(OUT_DIR)):
    fa_path = os.path.join(OUT_DIR, subj_folder, "FA.nii.gz")
    status = "COMPLETE" if os.path.exists(fa_path) else "incomplete"
    print(f"{subj_folder}: {status}")

In [ ]:
succeeded, failed = [], []

for i, subj in enumerate(NEW_SUBJECTS, 1):
    subj_out = os.path.join(OUT_DIR, subj)
    fa_path = os.path.join(subj_out, "FA.nii.gz")

    if os.path.exists(fa_path):
        print(f"[{i}/{len(NEW_SUBJECTS)}] {subj} already done - skipping.")
        succeeded.append(subj)
        continue

    print(f"\n=== [{i}/{len(NEW_SUBJECTS)}] Processing subject {subj} ===")
    try:
        download_subject(subj, os.path.join(RAW_DIR, subj))
        preprocess_subject(subj, RAW_DIR, OUT_DIR, use_gpu=True)
        succeeded.append(subj)
    except Exception as e:
        print(f"  [{subj}] FAILED: {e}")
        failed.append(subj)
    finally:
        shutil.rmtree(os.path.join(RAW_DIR, subj), ignore_errors=True)

print("\n" + "="*60)
print(f"DONE. Succeeded: {len(succeeded)}  Failed: {len(failed)}")
if failed:
    print("Failed subjects:", ", ".join(failed))
print("="*60)


## Step 9 - Quick visual sanity check on the first successful subject

In [ ]:
print("Succeeded:", succeeded)
print("Failed:", failed)


In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib

check_subj = succeeded[0] if succeeded else SUBJECTS[0]
subj_out = os.path.join(OUT_DIR, check_subj)

t1 = nib.load(os.path.join(subj_out, "T1_brain.nii.gz")).get_fdata()
fa = nib.load(os.path.join(subj_out, "FA.nii.gz")).get_fdata()

mid_slice = t1.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(t1[:, :, mid_slice].T, cmap="gray", origin="lower")
axes[0].set_title(f"{check_subj} - T1 (skull-stripped)")
axes[0].axis("off")

axes[1].imshow(fa[:, :, mid_slice].T, cmap="hot", origin="lower")
axes[1].set_title(f"{check_subj} - FA map")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("If the T1 shows a clean brain (no skull) and the FA map shows bright white matter tracts,")
print("the pipeline worked correctly.")


## Step 10 - Save your outputs

`/kaggle/working/` is wiped when the session ends unless you click **Save Version** (top right) before closing — do this periodically during a long run, not just at the end.

Alternatively, zip and download `processed_hcp/` directly:

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/processed_hcp", "zip", OUT_DIR)
print("Zipped to /kaggle/working/processed_hcp.zip - download it from the Output pane on the right.")
